# Generate the LCA impact scores

In [1]:
# Import the required libraries
from mescal import *
import pandas as pd
import bw2data as bd

## Set up and input data

In [2]:
ecoinvent_version = '3.12' # choose the ecoinvent version you wish to use
regionalize_foregrounds = ['Operation', 'Resource'] # set to 'all' if you want to regionalize the foreground inventories of all types of LCI datasets
premise_iam = 'remind-eu' # choose the IAM to which the premise database is linked
premise_ssp_rcp = 'SSP2-NDC' # choose the SSP/RCP scenario to which the premise database is linked
years = [2030, 2040, 2050] # choose the years for which you want to generate metrics

In [3]:
# Set the name of the new database with the ESM datasets (the different years will be added to that name)
esm_db_name = 'PyPSA-Eur'

In [4]:
# Main version of ecoinvent (without .1 if any)
ecoinvent_main_version = '.'.join(ecoinvent_version.split('.')[:2])

In [5]:
lcia_methods = [
    'EF v3.1',
    'IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
]

In [6]:
# Set up your Brightway project
bd.projects.set_current('pypsa') # put the name of your brightway project here

In [7]:
# Name of the biosphere database of your project
biosphere_db_name = 'biosphere3'

In [8]:
# Input data files for all time steps
input_data_folder = 'your_specific_input_files'
mapping = pd.read_csv(f'{input_data_folder}/mapping.csv')
unit_conversion = pd.read_csv(f'{input_data_folder}/unit_conversion.csv')
mapping_esm_flows_to_CPC = pd.read_csv(f'{input_data_folder}/mapping_esm_flows_to_CPC.csv')
technology_compositions = pd.read_csv(f'{input_data_folder}/technology_compositions.csv')
technology_specifics = pd.read_csv(f'{input_data_folder}/technology_specifics.csv')
efficiency = pd.read_csv(f'{input_data_folder}/efficiency.csv')
impact_abbrev = pd.read_csv('generic_input_files/impact_abbrev.csv')
mapping_new_products_to_cpc = pd.read_csv('generic_input_files/mapping_new_products_to_CPC.csv')

In [9]:
def create_custom_dataset_for_process_emissions(db_name):
    code = random_code()

    return {
        'name': 'carbon dioxide emissions from processes',
        'reference product': 'carbon dioxide, emitted',
        'unit': 'kilogram',
        'location': 'GLO',
        'exchanges': [
            {
                'name': 'carbon  dioxide, emitted',
                'amount': 1.0,
                'unit': 'kilogram',
                'type': 'production',
                'location': 'GLO',
                'code': code,
                'database': db_name,
            },
            {
                'input': (biosphere_db_name, '349b29d1-3e58-4c66-98b9-9d1a076efd2e'),
                'name': 'Carbon dioxide, fossil',
                'amount': 1.0,
                'unit': 'kilogram',
                'database': biosphere_db_name,
                'code': '349b29d1-3e58-4c66-98b9-9d1a076efd2e',
                'type': 'biosphere',
                'categories': ('air',),
            }
        ],
        'comment': 'Custom dataset for the CO2 emissions of processes in PyPSA',
        'database': db_name,
        'code': code,
        'classifications': [('CPC', '0000: Custom CPC')],
    }

In [10]:
# Time-dependent input data files
time_steps = []
for index, year in enumerate(years):
    model = pd.read_csv(f'{input_data_folder}/{year}/model.csv')
    lifetime = pd.read_csv(f'{input_data_folder}/{year}/lifetime.csv')
    main_database_name = f"ecoinvent_cutoff_{ecoinvent_version}_{premise_iam}_{premise_ssp_rcp}_{year}"
    main_database = Database(main_database_name, create_pickle=True)
    process_emissions_ds = create_custom_dataset_for_process_emissions(main_database_name)
    main_database += Database(db_as_list=[process_emissions_ds])
    main_database.add_CPC_categories(
        overwrite_existing_CPC=True, 
        mapping_new_products_to_CPC=mapping_new_products_to_cpc,
        )  # the CPC categories of CO2 flows need to be overwritten (same category as H2)
    time_steps.append({'year': year, 'model': model, 'lifetime': lifetime, 'main_database': main_database, 'main_database_name': main_database_name})

    # The database name in the mapping file should be the one of the first time step
    if index == 0:
        mapping.Database = main_database_name

2026-11-11 09:20:47,760 - Database - INFO - Loaded ecoinvent_cutoff_3.12_remind-eu_SSP2-NDC_2030 from pickle!
2026-11-11 09:20:53,559 - Database - INFO - Loaded ecoinvent_cutoff_3.12_remind-eu_SSP2-NDC_2040 from pickle!
2026-11-11 09:20:59,237 - Database - INFO - Loaded ecoinvent_cutoff_3.12_remind-eu_SSP2-NDC_2050 from pickle!


## Initialize the PathwayESM class

In [11]:
esm = PathwayESM(
    time_steps=time_steps,
    mapping=mapping,
    unit_conversion=unit_conversion,
    mapping_esm_flows_to_CPC_cat=mapping_esm_flows_to_CPC,
    esm_db_name=esm_db_name,
    biosphere_db_name=biosphere_db_name,
    esm_location='RER',
    technology_compositions=technology_compositions,
    tech_specifics=technology_specifics,
    efficiency=efficiency,
    regionalize_foregrounds=regionalize_foregrounds,
    accepted_locations=['RER', 'WEU', 'EWN'],
    locations_ranking=['WEU', 'CEU', 'EWN', 'RER', 'Europe without Switzerland', 'CH', 'DE', 'RoW', 'GLO'],
)

In [12]:
esm.change_location_mapping_file()

In [13]:
missing_flows = time_steps[0]['main_database'].test_mapping_file(esm.mapping[esm.mapping.Year == years[0]])

2026-11-11 09:21:19,984 - Database - INFO - Mapping successfully linked to the database


In [14]:
esm.check_inputs()

2026-11-11 09:21:20,093 - Mescal - WARNING - List of technologies or resources that are in the mapping file but not in the model file (this will not be a problem in the workflow): ['AC (Line)', 'H2 Store', 'H2 pipeline', 'battery', 'co2 stored (Store)', 'gas (Store)', 'gas pipeline', 'home battery (Store)', 'methanol (Store)', 'oil (Store)', 'rural water tanks (Store)', 'urban central water pits (Store)', 'urban central water tanks (Store)', 'urban decentral water tanks (Store)']
2026-11-11 09:21:20,231 - Mescal - WARNING - List of flows that are in the model file but not in the ESM flows to CPC mapping file. It might be an issue for double counting if these flows are inputs of some ESM technologies: ['solid biomass import']
2026-11-11 09:21:55,556 - Mescal - WARNING - Some technologies have no lifetime value for LCA in the lifetime file. Therefore, lifetime harmonization with the ESM will not be performed during the LCIA phase and capacity factor harmonization during the feedback of E

## Create the ESM databases

In [15]:
esm.create_esm_database()

2026-11-11 09:22:01,801 - Mescal - INFO - Starting to remove double-counted flows
100%|██████████| 57/57 [00:05<00:00, 10.84it/s]
2026-11-11 09:22:08,515 - Mescal - INFO - Double-counting removal done in 6.7 seconds
2026-11-11 09:22:08,571 - Mescal - WARNING - No construction input flow was found in the following operation datasets (this is not necessarily an issue, but you might want to have a look): ['urban central resistive heater']
2026-11-11 09:22:08,575 - Mescal - WARNING - Some input flows could not be found in the following operation datasets (this might be due to the operation dataset being a proxy of the technology or to an unmapped CPC category): {'DAC': ['co2'], 'HVC to air': ['non-sequestered HVC'], 'biogas to gas': ['co2'], 'process emissions (Link)': ['process emissions'], 'process emissions CC': ['process emissions', 'waste heat'], 'solid biomass for industry CC': ['co2'], 'urban central solid biomass CHP CC': ['co2'], 'land transport EV (Load)': ['EV battery']}
2026-11

Title: Writing activities to SQLite3 database:
  Started: 11/11/2026 09:22:59
  Finished: 11/11/2026 09:22:59
  Total time elapsed: 00:00:00
  CPU %: 85.20
  Memory %: 22.82


2026-11-11 09:24:18,754 - Database - INFO - PyPSA-Eur_2030 written to Brightway!
2026-11-11 09:24:19,917 - Mescal - INFO - Database written in 128.4 seconds
2026-11-11 09:24:25,607 - Mescal - INFO - Starting to remove double-counted flows
100%|██████████| 57/57 [00:04<00:00, 11.43it/s]
2026-11-11 09:24:31,746 - Mescal - INFO - Double-counting removal done in 6.1 seconds
2026-11-11 09:24:31,802 - Mescal - WARNING - No construction input flow was found in the following operation datasets (this is not necessarily an issue, but you might want to have a look): ['urban central resistive heater']
2026-11-11 09:24:31,805 - Mescal - WARNING - Some input flows could not be found in the following operation datasets (this might be due to the operation dataset being a proxy of the technology or to an unmapped CPC category): {'DAC': ['co2'], 'HVC to air': ['non-sequestered HVC'], 'biogas to gas': ['co2'], 'process emissions (Link)': ['process emissions'], 'process emissions CC': ['process emissions'

Title: Writing activities to SQLite3 database:
  Started: 11/11/2026 09:25:15
  Finished: 11/11/2026 09:25:15
  Total time elapsed: 00:00:00
  CPU %: 99.40
  Memory %: 22.72


2026-11-11 09:26:35,316 - Database - INFO - PyPSA-Eur_2040 written to Brightway!
2026-11-11 09:26:37,050 - Mescal - INFO - Database written in 122.1 seconds
2026-11-11 09:26:42,855 - Mescal - INFO - Starting to remove double-counted flows
100%|██████████| 57/57 [00:05<00:00, 10.78it/s]
2026-11-11 09:26:49,378 - Mescal - INFO - Double-counting removal done in 6.5 seconds
2026-11-11 09:26:49,542 - Mescal - WARNING - No construction input flow was found in the following operation datasets (this is not necessarily an issue, but you might want to have a look): ['urban central resistive heater']
2026-11-11 09:26:49,547 - Mescal - WARNING - Some input flows could not be found in the following operation datasets (this might be due to the operation dataset being a proxy of the technology or to an unmapped CPC category): {'DAC': ['co2'], 'HVC to air': ['non-sequestered HVC'], 'biogas to gas': ['co2'], 'process emissions (Link)': ['process emissions'], 'process emissions CC': ['process emissions'

Title: Writing activities to SQLite3 database:
  Started: 11/11/2026 09:27:31
  Finished: 11/11/2026 09:27:31
  Total time elapsed: 00:00:00
  CPU %: 149.60
  Memory %: 14.81


2026-11-11 09:28:34,271 - Database - INFO - PyPSA-Eur_2050 written to Brightway!
2026-11-11 09:28:35,943 - Mescal - INFO - Database written in 102.9 seconds


## Compute the impact scores

In [16]:
# Life-cycle impact scores
impact_scores, contrib_analysis, _ = esm.compute_impact_scores(
    methods=lcia_methods,
    contribution_analysis='both',
    contribution_analysis_limit_type='number',
    contribution_analysis_limit=10,
)
impact_scores.to_csv(f'{esm.results_path_file}/impact_scores.csv', index=False)
contrib_analysis.to_csv(f'{esm.results_path_file}/contribution_analysis.csv', index=False)

Getting activity data


100%|██████████| 149/149 [00:00<00:00, 49579.63it/s]


Adding exchange data to activities


100%|██████████| 3011/3011 [00:00<00:00, 46618.34it/s]


Filling out exchange data


100%|██████████| 149/149 [00:00<00:00, 254.41it/s]
2026-11-11 09:28:37,245 - Database - INFO - Loaded PyPSA-Eur_2030 from brightway!


... (26 outputs left out)

Adding exchange data to activities


100%|██████████| 3011/3011 [00:00<00:00, 76185.55it/s]


Filling out exchange data


100%|██████████| 149/149 [00:00<00:00, 730.38it/s]
2026-11-11 09:34:03,557 - Database - INFO - Loaded PyPSA-Eur_2050 from brightway!
148it [02:54,  1.18s/it]


In [17]:
# Direct impact scores
impact_scores_direct, _, _ = esm.compute_impact_scores(
    methods=lcia_methods,
    assessment_type='direct emissions',
)
impact_scores_direct.to_csv(f'{esm.results_path_file}/impact_scores_direct.csv', index=False)

Getting activity data


100%|██████████| 149/149 [00:00<00:00, 149081.89it/s]


Adding exchange data to activities


100%|██████████| 3011/3011 [00:00<00:00, 31188.07it/s]


Filling out exchange data


100%|██████████| 149/149 [00:00<00:00, 322.09it/s]
2026-11-11 09:37:10,270 - Database - INFO - Loaded PyPSA-Eur_2030 from brightway!


... (32 outputs left out)

Filling out exchange data


100%|██████████| 149/149 [00:00<00:00, 535.53it/s]
2026-11-11 09:37:16,962 - Database - INFO - Loaded PyPSA-Eur_2050 from brightway!
Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 11/11/2026 09:37:18
  Finished: 11/11/2026 09:37:18
  Total time elapsed: 00:00:00
  CPU %: 84.90
  Memory %: 17.45


2026-11-11 09:37:19,257 - Database - INFO - PyPSA-Eur_2050_direct_emissions written to Brightway!
57it [00:01, 43.93it/s]


## Comparison between PyPSA's direct CO2 emissions and direct GHG emissions from LCI datasets

In [18]:
impact_scores_direct = pd.read_csv(f'{esm.results_path_file}/impact_scores_direct.csv')

In [19]:
impact_scores_direct_co2 = impact_scores_direct[
    (impact_scores_direct.Impact_category == "('IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total')")
].copy()

impact_scores_direct_co2['Value'] *= 1e-3  # from kg CO2-eq to t CO2-eq

In [20]:
esm.validation_direct_carbon_emissions(
    R_direct=impact_scores_direct_co2,
    lcia_method_carbon_emissions=('IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total'),
    carbon_flow_in_esm='co2',
)